In [1]:
# Pipeline Configuration
RUN_MODE = "production"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 4
CHUNK_SIZE = 500
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = False
PIPELINE_VERSION = "1.0.0"


# !pip -q install geopandas rasterio rioxarray pystac-client planetary-computer odc-stac shapely pyproj xarray folium leafmap




In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point

import rasterio
import rioxarray

import planetary_computer
import pystac_client




In [4]:
import os

folders = [
    "../data",
    "../data/raw",
    "../data/processed",
    "../data/features",
    "../data/metadata",
    "../data/final",
    "../data/lucas",
    "../data/sentinel",
    "../data/weather",
    "../data/soilgrids",
    "../outputs",
    "../outputs/csv",
    "../outputs/maps",
    "../outputs/figures",
    "../outputs/reports",
    "../outputs/metrics",
    "../outputs/learning_curves",
    "../outputs/feature_importance",
    "../outputs/confusion_matrix",
    "../models",
    "../models/machine_learning",
    "../models/deep_learning",
    "../models/ensemble",
    "../models/best",
    "../models/experimental"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")





Created: ../data
Created: ../data/raw
Created: ../data/processed
Created: ../data/features
Created: ../data/metadata
Created: ../data/final
Created: ../data/lucas
Created: ../data/sentinel
Created: ../data/weather
Created: ../data/soilgrids
Created: ../outputs
Created: ../outputs/csv
Created: ../outputs/maps
Created: ../outputs/figures
Created: ../outputs/reports
Created: ../outputs/metrics
Created: ../outputs/learning_curves
Created: ../outputs/feature_importance
Created: ../outputs/confusion_matrix
Created: ../models
Created: ../models/machine_learning
Created: ../models/deep_learning
Created: ../models/ensemble
Created: ../models/best
Created: ../models/experimental


In [5]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected Successfully")




Connected Successfully


# # collections = catalog.get_collections()

# # for c in collections:
    # # print(c.id)



print("Skipping collections listing for speed.")




from shapely.geometry import Point

lon = 31.083
lat = 30.563

point = Point(lon, lat)

buffer = point.buffer(0.001)

geometry = buffer.__geo_interface__

geometry




**=================================================================================**

# !pip -q install cdsapi xarray netCDF4




# url: https://cds.climate.copernicus.eu/api
# key: 006b1498-1c8f-40c7-b903-cfa0e43acdd8




import os

token = "006b1498-1c8f-40c7-b903-cfa0e43acdd8"

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {token}\n")




import cdsapi

client = cdsapi.Client()
print("Connected Successfully")




import cdsapi

client = cdsapi.Client()

print("Connected")




import cdsapi

client = cdsapi.Client()

client.retrieve(
    "reanalysis-era5-land",
    {
        "variable": "2m_temperature",
        "year": "2024",
        "month": "05",
        "day": "22",
        "time": "12:00",
        "data_format": "netcdf",
        "download_format": "unarchived"
    },
    "../data/raw/era5/test.nc"
)

print("Downloaded Successfully")




import cdsapi

client = cdsapi.Client()

# مركز المزرعة
lat = 30.563
lon = 31.083

# هامش 0.05 درجة (~5.5 كم)
delta = 0.05

client.retrieve(
    "reanalysis-era5-land",
    {
        "variable": [
            "2m_temperature",
            "total_precipitation",
            "surface_solar_radiation_downwards",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "volumetric_soil_water_layer_1",
            "soil_temperature_level_1"
        ],

        "year": "2024",
        "month": "05",
        "day": "22",

        "time": [
            "00:00","01:00","02:00","03:00",
            "04:00","05:00","06:00","07:00",
            "08:00","09:00","10:00","11:00",
            "12:00","13:00","14:00","15:00",
            "16:00","17:00","18:00","19:00",
            "20:00","21:00","22:00","23:00"
        ],

        # North, West, South, East
        "area": [
            lat + delta,
            lon - delta,
            lat - delta,
            lon + delta
        ],

        "data_format": "netcdf",
        "download_format": "unarchived"
    },

    "../data/raw/era5/era5.nc"
)

print("Download Finished")




import xarray as xr

ds = xr.open_dataset("../data/raw/era5/era5.nc")

print(ds)
print(ds.data_vars)




import numpy as np

# Temperature (Kelvin -> Celsius)
temperature = ds["t2m"].values - 273.15

# Rainfall (meter -> mm)
rainfall = ds["tp"].values * 1000

# Solar Radiation
solar = ds["ssrd"].values

# Wind
u = ds["u10"].values
v = ds["v10"].values

# Soil Moisture
soil_moisture = ds["swvl1"].values

# Soil Temperature
soil_temp = ds["stl1"].values - 273.15




wind_speed = np.sqrt(u**2 + v**2)

print("Mean Temperature :", np.mean(temperature))
print("Max Temperature  :", np.max(temperature))
print("Min Temperature  :", np.min(temperature))

print()

print("Total Rainfall   :", np.sum(rainfall))
print("Mean Rainfall    :", np.mean(rainfall))

print()

print("Mean Soil Moisture :", np.mean(soil_moisture))
print("Max Soil Moisture  :", np.max(soil_moisture))

print()

print("Mean Soil Temp :", np.mean(soil_temp))

print()

print("Mean Solar Radiation :", np.mean(solar))

print()

print("Mean Wind Speed :", np.mean(wind_speed))
print("Max Wind Speed  :", np.max(wind_speed))




features = {

    # Weather
    "temperature_mean": float(ds.t2m.mean()-273.15),
    "temperature_max": float(ds.t2m.max()-273.15),
    "temperature_min": float(ds.t2m.min()-273.15),

    "rainfall_total_mm": float(ds.tp.sum()*1000),
    "rainfall_mean_mm": float(ds.tp.mean()*1000),

    "soil_moisture_mean": float(ds.swvl1.mean()),
    "soil_moisture_max": float(ds.swvl1.max()),

    "soil_temperature_mean": float(ds.stl1.mean()-273.15),

    "solar_radiation_mean":
        float(ds.ssrd.mean()/86400),

    "wind_speed_mean":
        float(np.sqrt(ds.u10**2 + ds.v10**2).mean()),

    "wind_speed_max":
        float(np.sqrt(ds.u10**2 + ds.v10**2).max())
}


features




In [19]:
import os
import json
import time
import glob
import datetime
import gc
import pandas as pd
import numpy as np
import cdsapi
import xarray as xr

folders = ["../data/raw/weather", "../data/metadata", "../logs"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

client = cdsapi.Client()
df_lucas = pd.read_csv("../data/features/lucas_labels.csv")
checkpoint_file = "../data/metadata/weather_checkpoint.json"
chunk_dir = "../data/features/weather_chunks"
os.makedirs(chunk_dir, exist_ok=True)
failed_points_file = "../data/metadata/failed_points.csv"
log_file = "../logs/weather.log"

def log_msg(msg):
    timestamp = datetime.datetime.now().isoformat()
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {msg}\n")
    print(msg)

start_idx = 0
if ENABLE_CHECKPOINT and os.path.exists(checkpoint_file):
    try:
        with open(checkpoint_file, "r") as f:
            cp = json.load(f)
            start_idx = cp.get("last_processed_idx", -1) + 1
        log_msg(f"Resuming Weather from index {start_idx}")
    except Exception as e:
        log_msg(f"Failed to read checkpoint: {e}")

def get_local_weather_box(lat, lon, date_str):
    grid_lat = int(lat)
    grid_lon = int(lon)
    dt = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
    if pd.isna(dt):
        year = '2018'
        month = '07'
    else:
        year = str(dt.year)
        month = f'{dt.month:02d}'
    local_path = f"../data/raw/weather/weather_grid_{grid_lat}_{grid_lon}_{year}_{month}.nc"
    if ENABLE_CACHE and os.path.exists(local_path):
        return local_path
    area = [grid_lat + 1, grid_lon, grid_lat, grid_lon + 1]
    for attempt in range(MAX_RETRIES):
        try:
            client.retrieve(
                "reanalysis-era5-land",
                {
                    "variable": [
                        "2m_temperature",
                        "total_precipitation",
                        "volumetric_soil_water_layer_1",
                        "soil_temperature_level_1",
                        "surface_net_solar_radiation",
                        "10m_u_component_of_wind",
                        "10m_v_component_of_wind"
                    ],
                    "year": year,
                    "month": month,
                    "day": ["01","02","03","04","05","06","07","08","09","10","11","12","13","14","15","16","17","18","19","20","21","22","23","24","25","26","27","28"],
                    "time": ["00:00", "06:00", "12:00", "18:00"],
                    "area": area,
                    "format": "netcdf"
                },
                local_path + ".tmp"
            )
            # Check if downloaded file is a ZIP archive
            with open(local_path + ".tmp", "rb") as f_obj:
                sig = f_obj.read(4)
            if sig == b'PK\x03\x04':
                import zipfile, shutil
                tmp_dir = local_path + "_unzip"
                os.makedirs(tmp_dir, exist_ok=True)
                with zipfile.ZipFile(local_path + ".tmp", "r") as zip_ref:
                    zip_ref.extractall(tmp_dir)
                nc_files = [x for x in os.listdir(tmp_dir) if x.endswith('.nc')]
                if nc_files:
                    shutil.move(os.path.join(tmp_dir, nc_files[0]), local_path)
                shutil.rmtree(tmp_dir)
                os.remove(local_path + ".tmp")
            else:
                os.rename(local_path + ".tmp", local_path)
            return local_path
        except Exception as e:
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to download weather grid for {grid_lat},{grid_lon} month {month}")

def process_point(row):
    point_id = int(row["POINT_ID"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    date_str = str(row["Survey_Date"])
    meta = {
        "POINT_ID": point_id, "Latitude": lat, "Longitude": lon, "Survey_Date": date_str,
        "Extraction_Time": datetime.datetime.now().isoformat(), "Pipeline_Version": PIPELINE_VERSION,
        "ERA5_Time": date_str[:7],
        "weather_ok": np.nan
    }
    try:
        if abs(lat - 30.563) < 1e-4 and abs(lon - 31.083) < 1e-4:
            fpath = "../data/raw/era5/era5.nc"
        else:
            fpath = get_local_weather_box(lat, lon, date_str)
        with xr.open_dataset(fpath) as ds:
            ds_point = ds.sel(latitude=lat, longitude=lon, method="nearest")
            t2m = ds_point["t2m"].values - 273.15
            tp = ds_point["tp"].values * 1000.0
            ssrd = ds_point["ssrd"].values if "ssrd" in ds_point else ds_point["ssr"].values
            u10 = ds_point["u10"].values
            v10 = ds_point["v10"].values
            swvl1 = ds_point["swvl1"].values
            stl1 = ds_point["stl1"].values - 273.15
            wind = np.sqrt(u10**2 + v10**2)
            features = {
                "temperature_mean": float(t2m.mean()),
                "temperature_max": float(t2m.max()),
                "temperature_min": float(t2m.min()),
                "rainfall_total_mm": float(tp.sum()),
                "rainfall_mean_mm": float(tp.mean()),
                "soil_moisture_mean": float(swvl1.mean()),
                "soil_moisture_max": float(swvl1.max()),
                "soil_temperature_mean": float(stl1.mean()),
                "solar_radiation_mean": float((ssrd / 86400).mean()),
                "wind_speed_mean": float(wind.mean()),
                "wind_speed_max": float(wind.max())
            }
            meta.update(features)
            meta["weather_ok"] = 1.0
            meta["CRS"] = "EPSG:4326"
            return {"status": "success", "data": meta}
    except Exception as e:
        err_msg = str(e)
        failed_row = {
            "POINT_ID": point_id, "longitude": lon, "latitude": lat, "date": date_str,
            "error_message": err_msg, "timestamp": datetime.datetime.now().isoformat()
        }
        pd.DataFrame([failed_row]).to_csv(
            failed_points_file, mode="a", header=not os.path.exists(failed_points_file), index=False
        )
        return {"status": "failed", "point_id": point_id, "error": err_msg}

num_samples = len(df_lucas)
start_time = time.time()
for c_idx in range(start_idx, num_samples, CHUNK_SIZE):
    chunk = df_lucas.iloc[c_idx : min(c_idx + CHUNK_SIZE, num_samples)]
    chunk_data = []
    log_msg(f"Weather Processing batch {c_idx} to {min(c_idx+CHUNK_SIZE, num_samples)}...")
    for _, row in chunk.iterrows():
        res = process_point(row)
        if res["status"] == "success":
            chunk_data.append(res["data"])
    if chunk_data:
        df_chunk = pd.DataFrame(chunk_data).drop_duplicates(subset=["POINT_ID"])
        df_chunk.to_csv(f"{chunk_dir}/weather_part_{c_idx//CHUNK_SIZE + 1:03d}.csv", index=False)
    if ENABLE_CHECKPOINT:
        with open(checkpoint_file, "w") as f:
            json.dump({"last_processed_idx": min(c_idx + CHUNK_SIZE - 1, num_samples - 1)}, f)
    elapsed = time.time() - start_time
    processed = min(c_idx + CHUNK_SIZE, num_samples) - start_idx
    remaining = num_samples - min(c_idx + CHUNK_SIZE, num_samples)
    eta = (elapsed / processed) * remaining if processed > 0 else 0
    log_msg(f"Progress Weather: {min(c_idx+CHUNK_SIZE, num_samples)}/{num_samples} ({processed/num_samples*100:.1f}%) | ETA: {eta/60:.1f} min")
    del chunk_data
    gc.collect()

all_chunks = sorted(glob.glob(f"{chunk_dir}/weather_part_*.csv" ))
if all_chunks:
    df_final = pd.concat([pd.read_csv(ch) for ch in all_chunks], ignore_index=True)
    df_final.to_csv("../data/features/weather_features.csv", index=False)
    log_msg(f"Weather Merge Complete. Total rows: {len(df_final)}")





import os
import json
import time
import glob
import datetime
import gc
import pandas as pd
import numpy as np
import cdsapi
import xarray as xr

client = cdsapi.Client()

def process_point(row):
    point_id = int(row["POINT_ID"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    date_str = str(row["Survey_Date"])
    meta = {
        "POINT_ID": point_id, "Latitude": lat, "Longitude": lon, "Survey_Date": date_str,
        "Extraction_Time": datetime.datetime.now().isoformat(), "Pipeline_Version": PIPELINE_VERSION,
        "ERA5_Time": date_str[:7],
        "weather_ok": np.nan
    }
    try:
        if abs(lat - 30.563) < 1e-4 and abs(lon - 31.083) < 1e-4:
            fpath = "../data/raw/era5/era5.nc"
        else:
            fpath = None
            last_err = None
            for retry in range(MAX_RETRIES + 1):
                try:
                    fpath = get_local_weather_box(lat, lon, date_str)
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(2 ** retry)
            if fpath is None:
                raise last_err

        with xr.open_dataset(fpath) as ds:
            ds_point = ds.sel(latitude=lat, longitude=lon, method="nearest")
            t2m = ds_point["t2m"].values - 273.15
            tp = ds_point["tp"].values * 1000.0
            ssrd = ds_point["ssrd"].values if "ssrd" in ds_point else ds_point["ssr"].values
            u10 = ds_point["u10"].values
            v10 = ds_point["v10"].values
            swvl1 = ds_point["swvl1"].values
            stl1 = ds_point["stl1"].values - 273.15
            wind = np.sqrt(u10**2 + v10**2)
            features = {
                "temperature_mean": float(t2m.mean()),
                "temperature_max": float(t2m.max()),
                "temperature_min": float(t2m.min()),
                "rainfall_total_mm": float(tp.sum()),
                "rainfall_mean_mm": float(tp.mean()),
                "soil_moisture_mean": float(swvl1.mean()),
                "soil_moisture_max": float(swvl1.max()),
                "soil_temperature_mean": float(stl1.mean()),
                "solar_radiation_mean": float((ssrd / 86400).mean()),
                "wind_speed_mean": float(wind.mean()),
                "wind_speed_max": float(wind.max())
            }
            meta.update(features)
            meta["weather_ok"] = 1.0
            meta["CRS"] = "EPSG:4326"
            return {"status": "success", "data": meta}
    except Exception as e:
        err_msg = str(e)
        features = {
            "temperature_mean": np.nan, "temperature_max": np.nan, "temperature_min": np.nan,
            "rainfall_total_mm": np.nan, "rainfall_mean_mm": np.nan, "soil_moisture_mean": np.nan,
            "soil_moisture_max": np.nan, "soil_temperature_mean": np.nan, "solar_radiation_mean": np.nan,
            "wind_speed_mean": np.nan, "wind_speed_max": np.nan
        }
        meta.update(features)
        meta["weather_ok"] = 0.0
        meta["CRS"] = np.nan
        meta["ERA5_Time"] = np.nan
        return {"status": "success", "data": meta}


Weather Processing batch 0 to 10...
Progress Weather: 10/10 (100.0%) | ETA: 0.0 min


Weather Merge Complete. Total rows: 10
